# Integração das bases

Nesta etapa, as bases tratadas de Censo Escolar, rendimento e média de alunos por turma são integradas pela chave `ano + codigo_escola`, formando uma base analítica única por escola e ano.

## Integração das bases tratadas

As bases de Censo Escolar, rendimento e média de alunos por turma são integradas pela chave `ano + codigo_escola`. Antes da junção, são verificadas duplicidades e a cobertura das chaves para garantir consistência na construção da base analítica.

In [2]:
bases = {
    "Censo": censo,
    "Rendimento": rendimento,
    "Média de alunos": media_alunos
}

for nome, df in bases.items():
    duplicados = df.duplicated(
        subset=["ano", "codigo_escola"]
    ).sum()

    print(f"{nome}: {duplicados} chaves duplicadas")


for df in [censo, rendimento, media_alunos]:
    df["ano"] = pd.to_numeric(
        df["ano"],
        errors="coerce"
    ).astype("Int64")

    df["codigo_escola"] = (
        df["codigo_escola"]
        .astype("string")
        .str.strip()
    )


base_analitica = censo.copy()


rendimento_merge = rendimento[
    [
        "ano",
        "codigo_escola",
        "taxa_aprovacao",
        "taxa_reprovacao",
        "taxa_abandono",
    ]
].copy()



base_analitica = base_analitica.merge(
    rendimento_merge,
    on=["ano", "codigo_escola"],
    how="left",
    validate="one_to_one"
)


media_merge = media_alunos[
    [
        "ano",
        "codigo_escola",
        "media_alunos_fund",
        "media_alunos_ai",
        "media_alunos_af",
    ]
].copy()


base_analitica = base_analitica.merge(
    media_merge,
    on=["ano", "codigo_escola"],
    how="left",
    validate="one_to_one"
)


print("\nDimensão final:", base_analitica.shape)

print("\nCobertura dos indicadores:")
print(
    base_analitica.groupby("ano").agg(
        escolas=("codigo_escola", "size"),
        aprovacao_disponivel=("taxa_aprovacao", "count"),
        media_alunos_disponivel=("media_alunos_fund", "count"),
    )
)

print("\nNulos nas chaves:")
print(
    base_analitica[
        ["ano", "codigo_escola"]
    ].isna().sum()
)

display(base_analitica.head())

Censo: 0 chaves duplicadas
Rendimento: 0 chaves duplicadas
Média de alunos: 0 chaves duplicadas

Dimensão final: (1621, 51)

Cobertura dos indicadores:
      escolas  aprovacao_disponivel  media_alunos_disponivel
ano                                                         
2018      276                   268                      268
2019      272                   264                      264
2020      271                   263                      263
2021      268                   258                      260
2022      267                   259                      259
2023      267                   259                      259

Nulos nas chaves:
ano              0
codigo_escola    0
dtype: int64


,ano,uf,municipio,codigo_municipio,codigo_escola,nome_escola,dependencia_administrativa,localizacao,situacao_funcionamento,IN_AGUA_POTAVEL,...,QT_DOC_FUND_AF,QT_TUR_FUND,QT_TUR_FUND_AI,QT_TUR_FUND_AF,taxa_aprovacao,taxa_reprovacao,taxa_abandono,media_alunos_fund,media_alunos_ai,media_alunos_af
0,2018,RS,Porto Alegre,4314902,43000479,E E IND ENS FUN TUPE PAN,Estadual,1,1,0,...,2,2,0,2,91.7,8.3,0.0,6.0,NaN,NaN
1,2018,RS,Porto Alegre,4314902,43001254,E E IND ENS FUN PINDO POTY,Estadual,1,1,0,...,1,2,1,1,100.0,0.0,0.0,1.5,1.0,NaN
2,2018,RS,Porto Alegre,4314902,43005047,E E IND ENS FUN KA AGUY MIRI,Estadual,1,1,0,...,2,2,0,2,100.0,0.0,0.0,3.0,NaN,NaN
3,2018,RS,Porto Alegre,4314902,43048200,EMEF PORTO NOVO,Municipal,1,1,0,...,8,13,9,4,78.0,22.0,0.0,26.9,26.0,29.0
4,2018,RS,Porto Alegre,4314902,43104932,COLEGIO DE APLICACAO UFRGS,Federal,1,1,0,...,39,12,5,7,88.5,11.5,0.0,24.7,20.0,28.0


## Exportação da base analítica

A base integrada é salva em `data/processed` para utilização nas etapas de análise exploratória, construção de indicadores e modelagem.

In [3]:
base_analitica.to_csv(
    PROCESSED / "base_analitica_escola_ano.csv",
    index=False
)

print("Base analítica salva com sucesso.")

Base analítica salva com sucesso.
